# 04 · 前處理與 Pipeline

真實資料很少能直接餵給模型——數值尺度差很多、有文字類別欄位。這一課學會用 **StandardScaler** 標準化、用 **Pipeline** 把前處理和模型綁成一體，並避開新手最常踩的**資料洩漏**陷阱。

## 學習目標

- 理解為什麼很多模型需要**特徵標準化**
- 用 `StandardScaler` 標準化，並只在訓練集上 `fit`
- 用 `Pipeline` 把「前處理 + 模型」串成一個物件
- 用 `ColumnTransformer` + `OneHotEncoder` 處理混合型欄位

## 1. 為什麼要標準化？

像 KNN 這種**靠距離**的模型，特徵尺度會嚴重影響結果。假設一個特徵範圍是 0~1、另一個是 0~10000，那後者會完全主宰距離計算，前者等於被忽略。

**標準化**把每個特徵變成「平均 0、標準差 1」，讓大家站在同一個尺度上公平競爭。

In [1]:
import numpy as np
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

wine = load_wine()
X, y = wine.data, wine.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print("標準化前，前 3 個特徵的範圍差很大:")
print("  各特徵平均:", X_train[:, :3].mean(axis=0).round(2))
print("  各特徵標準差:", X_train[:, :3].std(axis=0).round(2))

scaler = StandardScaler().fit(X_train)   # 只 fit 訓練集！
X_train_scaled = scaler.transform(X_train)
print("\n標準化後:")
print("  各特徵平均:", X_train_scaled[:, :3].mean(axis=0).round(2))
print("  各特徵標準差:", X_train_scaled[:, :3].std(axis=0).round(2))

標準化前，前 3 個特徵的範圍差很大:
  各特徵平均: [12.97  2.33  2.37]
  各特徵標準差: [0.8  1.09 0.27]

標準化後:
  各特徵平均: [0. 0. 0.]
  各特徵標準差: [1. 1. 1.]


## 2. 致命陷阱：資料洩漏（data leakage）

注意上面 `scaler.fit(X_train)` —— **只用訓練集**算平均和標準差，**絕不碰測試集**。

如果你先對全部資料做標準化再切分，測試集的資訊（平均、標準差）就「洩漏」進了訓練過程。評估出來的分數會虛高，上線後現出原形。

> 鐵則：**任何 `fit` 都只能看訓練集。** 測試集只能被 `transform`，永遠不被 `fit`。

## 3. Pipeline：把步驟綁成一個物件

手動「先 scaler 再 model」很容易忘記、或不小心對測試集 `fit`。`Pipeline` 幫你把多個步驟串成單一 estimator——對它呼叫 `fit`，它會自動依序 `fit_transform` 前面每一步、最後 `fit` 模型；呼叫 `predict` 時則只 `transform`。**資料洩漏自動被擋掉**。

In [2]:
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier

# 不做標準化的 KNN（對照組）
raw_knn = KNeighborsClassifier().fit(X_train, y_train)

# 用 Pipeline 串「標準化 + KNN」
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier()),
])
pipe.fit(X_train, y_train)   # scaler 只在 train 上 fit，自動防洩漏

print(f"沒標準化的 KNN：{raw_knn.score(X_test, y_test):.1%}")
print(f"有標準化的 KNN：{pipe.score(X_test, y_test):.1%}")

沒標準化的 KNN：77.8%
有標準化的 KNN：93.3%


看到標準化讓準確率大幅提升了吧——這就是前處理的價值。而且整條 `pipe` 現在就是一個普通 estimator，可以直接丟進交叉驗證、GridSearch（後面幾課會用到）。

## 4. 混合型欄位：ColumnTransformer

真實資料常常**數值欄 + 文字類別欄**混在一起。數值要標準化，類別要做 **One-Hot 編碼**（把「紅/綠/藍」變成三個 0/1 欄位）。`ColumnTransformer` 讓你對不同欄位套不同處理。

In [3]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# 一份迷你的混合型資料
df = pd.DataFrame({
    "age":    [25, 42, 31, 58, 29],
    "income": [40, 80, 55, 95, 47],
    "city":   ["Taipei", "Tainan", "Taipei", "Kaohsiung", "Tainan"],
})

pre = ColumnTransformer([
    ("num", StandardScaler(), ["age", "income"]),          # 數值欄 → 標準化
    ("cat", OneHotEncoder(sparse_output=False), ["city"]),  # 類別欄 → One-Hot
])

result = pre.fit_transform(df)
print("轉換後（前 2 欄是標準化數值，後 3 欄是 city 的 One-Hot）:")
print(result.round(2))
print("\n產生的欄位名稱:", list(pre.get_feature_names_out()))

轉換後（前 2 欄是標準化數值，後 3 欄是 city 的 One-Hot）:
[[-1.01 -1.13  0.    0.    1.  ]
 [ 0.42  0.8   0.    1.    0.  ]
 [-0.5  -0.4   0.    0.    1.  ]
 [ 1.76  1.52  1.    0.    0.  ]
 [-0.67 -0.79  0.    1.    0.  ]]

產生的欄位名稱: ['num__age', 'num__income', 'cat__city_Kaohsiung', 'cat__city_Tainan', 'cat__city_Taipei']


`ColumnTransformer` 一樣可以放進 `Pipeline`，組成「混合前處理 + 模型」的完整流程——這正是第 08 課實戰會用的骨架。

## 小結

- 靠距離的模型（KNN、SVM…）對特徵尺度敏感，需要**標準化**。
- **資料洩漏鐵則**：任何 `fit` 只看訓練集，測試集只能 `transform`。
- `Pipeline` 把前處理 + 模型綁成一個 estimator，自動防洩漏。
- `ColumnTransformer` 對數值欄、類別欄分別套用不同前處理。

## 練習

1. 把 `pipe` 裡的 `KNeighborsClassifier()` 換成 `SVC()`，標準化對 SVM 的幫助有多大？
2. 在 `ColumnTransformer` 範例多加一個類別欄（例如 `gender`），確認 One-Hot 欄位數正確增加。

下一課，我們認真談**評估**：交叉驗證、混淆矩陣、precision / recall、ROC。